## PARALELIZANDO SVM EN C++ DESDE JUPYTER NOTEBOOK

- **Estructura básica del dato**: Estímulos necesarios para el aprendizaje que deben registrarse en colecciones de datos, por lo general arreglos.

- **Dataset**: Conjunto de datos que se toman como verdades y se usan como el conocimiento. Por lo general se usa un archivo de datos.

- **Modelo**: Operaciones matemáticas que transforman los datos y disponen de una nueva estructura para lograr con los nuevos estímulos una decisión, al aplicar una ecuación matemática.

***Conceptos clave de la estadística y el aprendizaje automático que son fundamentales para entender cómo funcionan y se optimizan los modelos.***

1. **Tasa de aprendizaje:** La tasa de aprendizaje es un hiperparámetro que controla cuán grande es el paso que da el modelo en cada iteración para ajustar sus parámetros durante el proceso de entrenamiento.

  **Analogía:** Imaginen que están escalando una montaña y la tasa de aprendizaje es el tamaño de sus pasos. Si los pasos son muy grandes, podrían saltarse la cima y caer por el otro lado (oscilando sin encontrar la mejor solución). Si los pasos son muy pequeños, el proceso será muy lento y podría llevar mucho tiempo llegar a la cima (la solución óptima).

  **Impacto en el modelo:**

  - Alta tasa de aprendizaje: Puede acelerar el entrenamiento, pero corre el riesgo de no converger o incluso divergir.
  - Baja tasa de aprendizaje: Conduce a una convergencia más estable, pero puede hacer que el entrenamiento sea extremadamente lento.

2. **Parámetros regulatorios (para evitar el sobreajuste - overfitting):** El sobreajuste ocurre cuando un modelo se ajusta demasiado bien a los datos de entrenamiento, capturando tanto las tendencias reales como el ruido aleatorio, lo que resulta en un mal desempeño con datos nuevos o de prueba.

  **Analogía:** Imaginen que están tratando de predecir el precio de una casa basándose en varios factores como el tamaño, la ubicación y el número de habitaciones. Si su modelo está sobreajustado, podría estar basando sus predicciones en detalles muy específicos de las casas en el conjunto de entrenamiento que no son representativos en general.

  **Parámetros regulatorios comunes:**

  **Regularización L1 y L2:** Agregan una penalización por tener coeficientes de parámetros grandes, lo que ayuda a evitar que el modelo se ajuste demasiado a los datos de entrenamiento.
  - L1 (Lasso): Tiende a hacer que algunos coeficientes sean exactamente cero, lo que puede llevar a modelos más simples y fáciles de interpretar.
  - L2 (Ridge): Distribuye los coeficientes más uniformemente sin hacerlos exactamente cero.
  - Dropout (en redes neuronales): Durante el entrenamiento, se "apagan" aleatoriamente ciertas neuronas en cada iteración, lo que obliga a la red a no depender demasiado de ninguna neurona en particular.

3. **Sesgo (Bias):** El sesgo es la diferencia entre las predicciones promedio de nuestro modelo y los valores reales que estamos tratando de predecir. Un alto sesgo indica que el modelo no está capturando correctamente las tendencias subyacentes de los datos.

  **Analogía:** Volviendo al ejemplo de las casas, si su modelo tiene un alto sesgo, podría subestimar o sobreestimar sistemáticamente los precios de las casas sin importar cuánto entrenen el modelo.

  **Relación con la varianza:**

  - **Sesgo alto, varianza baja:** El modelo es demasiado simple y no captura bien las complejidades de los datos (subajuste).
  - **Sesgo bajo, varianza alta:** El modelo captura demasiados detalles de los datos de entrenamiento y no generaliza bien (sobreajuste).
  - **Sesgo y varianza equilibrados:** El objetivo es encontrar un punto intermedio donde el modelo capture adecuadamente las tendencias de los datos sin ajustarse demasiado a los detalles específicos del conjunto de entrenamiento.

In [10]:
import pandas as pd

df = pd.read_csv('/content/datos_completos.csv')

print("Primeras filas:")
print(df.head())

print("\nInformación general:")
df.info()

print("\nResumen estadístico:")
print(df.describe(include='all'))

print("\nNombres de columnas:")
print(df.columns)

print("\nForma del DataFrame:")
print(df.shape)

print("\nValores nulos por columna:")
print(df.isnull().sum())


Primeras filas:
         id     diagnóstico  radio_promedio  textura_promedio  \
0    842302               M           17.99             10.38   
1    842517               M           20.57             17.77   
2  84300903               M           19.69             21.25   
3  84348301               M           11.42             20.38   
4  84358402               M           20.29             14.34   

   perímetro_promedio  área_promedio  suavidad_promedio  compacidad_promedio  \
0              122.80         1001.0            0.11840              0.27760   
1              132.90         1326.0            0.08474              0.07864   
2              130.00         1203.0            0.10960              0.15990   
3               77.58          386.1            0.14250              0.28390   
4              135.10         1297.0            0.10030              0.13280   

   concavidad_promedio  puntos_cóncavos_promedio  ...  textura_peor  \
0               0.3001                   

## Cabeceros

In [11]:
!mkdir -p /content/src

### Archivo de cabecera: Punto.h


In [12]:
%%writefile /content/src/Punto.h
#ifndef PUNTO_H
#define PUNTO_H

#include <vector>

class Punto {
public:
    std::vector<float> caracteristicas;  //las caracteristicas son los estimulos que generan el entorno del problema
    int clase;  //Clase es la decisión final o el grupo al que pertenecen los estimulos

    Punto(const std::vector<float>& caracteristicas, int clase)
        : caracteristicas(caracteristicas), clase(clase) {}
};

#endif


Writing /content/src/Punto.h


### Archivo de cuerpo: Punto.cpp

In [13]:
%%writefile /content/src/Punto.cpp
#include "Punto.h"
//No se requiere detallar código por lo simple de Punto.h

Writing /content/src/Punto.cpp


In [14]:
!g++ -c /content/src/Punto.cpp -o /content/src/Punto

### Archivo de cabecera: Datos.h

In [15]:
%%writefile /content/src/Datos.h
#ifndef DATOS_H
#define DATOS_H

#include <vector>
#include <string>
#include "Punto.h"

class Datos {
public:
    std::vector<std::string> etiquetas;  //Arreglo de nombre de estimulos - Variables
    std::vector<Punto> leerDatosCSV(const std::string& archivoCSV); //Metodo que extrae los datos del archivo
};

#endif


Writing /content/src/Datos.h


### Archivo de cuerpo: Datos.cpp

In [16]:
%%writefile /content/src/Datos.cpp
#include "Datos.h"

#include <fstream>
#include <sstream>
#include <iostream>

std::vector<Punto> Datos::leerDatosCSV(const std::string& archivoCSV) {
    std::vector<Punto> puntos;
    std::ifstream archivo(archivoCSV);

    if (archivo.is_open()) {
        std::string linea;
        // Leer la primera línea (encabezado) y almacenar las etiquetas
        std::getline(archivo, linea);
        std::stringstream headerStream(linea);
        std::string etiqueta;
        std::getline(headerStream, etiqueta, ','); // id
        std::getline(headerStream, etiqueta, ','); // diagnosis
        while (std::getline(headerStream, etiqueta, ',')) {  //caracteristicas
            etiquetas.push_back(etiqueta);
        }

        // Leer los datos
        while (std::getline(archivo, linea)) {
            std::stringstream lineStream(linea);
            std::string cell;
            std::vector<float> caracteristicas;
            int clase;

            // Leer id y diagnóstico
            std::getline(lineStream, cell, ','); // id
            std::getline(lineStream, cell, ','); // diagnóstico
            clase = (cell == "M") ? 1 : -1;  //Maligno - Benigno

            // Leer las características
            while (std::getline(lineStream, cell, ',')) {
                caracteristicas.push_back(std::stof(cell));
            }

            // Crear un nuevo punto de datos y agregarlo al vector
            puntos.emplace_back(caracteristicas, clase);
        }
        archivo.close();
    } else {
        // Colores para la salida
        const std::string rojo = "\033[41;30m";
        const std::string reset = "\033[0m";

        std::cerr << rojo << "Error al abrir el archivo CSV: " << archivoCSV << reset << std::endl;
    }

    return puntos;
}


Writing /content/src/Datos.cpp


In [17]:
!g++ -c /content/src/Datos.cpp -o /content/src/Datos

## SVM en Serie

### Archivo de cabecera: SVM.h

In [18]:
%%writefile /content/src/SVM.h
#ifndef SVM_H
#define SVM_H

#include <vector>
#include <string>
#include "Punto.h"

class SVM {
private:
    std::vector<float> weights;  //vector de pesos, son los estimulos
    float bias;  //en la información del sesgo
    float learningRate;  //tasa de aprendizaje
    float regularizationParam;  //overfitting

public:
    SVM(float learningRate, float regularizationParam, int numCaracteristicas);

    void entrenar(const std::vector<Punto>& puntos, int epochs, bool mostrarProgreso = false);  //metodo para entrenar
    int clasificar(const Punto& puntoNuevo) const; //metodo para clasificar
    void imprimirParametros(const std::vector<std::string>& etiquetas, bool mostarResumen = false) const;  //metodo para inscribir los parametros
};

#endif


Writing /content/src/SVM.h


### Archivo del cuerpo: SVM.cpp

In [19]:
%%writefile /content/src/SVM.cpp
#include "SVM.h"
#include <iostream>
#include <iomanip>
#include <chrono>
#include <stdexcept>
#include <cmath>
#include <thread>
#include <ctime>  // Para std::chrono::milliseconds
#include <sys/resource.h> // Para obtener el uso de memoria

// Secuencia de escape para colores ANSI
#define ANSI_COLOR_GREEN   "\x1b[32m"
#define ANSI_COLOR_RESET   "\x1b[0m"

SVM::SVM(float learningRate, float regularizationParam, int numCaracteristicas)
    : learningRate(learningRate), regularizationParam(regularizationParam), bias(0.0) {
    if (numCaracteristicas <= 0) {
        throw std::invalid_argument("El número de características debe ser mayor que 0.");
    }
    weights = std::vector<float>(numCaracteristicas, 0.0); // Inicializar pesos para las características
}

void SVM::entrenar(const std::vector<Punto>& puntos, int epochs, bool mostrarProgreso) {
    if (epochs <= 0) {
        throw std::invalid_argument("El número de épocas debe ser mayor que 0.");
    }
    if (puntos.empty()) {
        throw std::invalid_argument("El conjunto de puntos no debe estar vacío.");
    }

    auto start = std::chrono::high_resolution_clock::now(); // Marca de tiempo de inicio

    if (mostrarProgreso) {
        std::cout << "ENTRENANDO EL MODELO" << "\n--------------------" << std::endl;
    }

    //ajustar el modelo por iteraciones
    for (int epoch = 0; epoch < epochs; ++epoch) {
        int correct = 0; // Para contar el número de clasificaciones correctas

        //- - - - - [ OPERACIONES MATETICAS DEL MODELO ] - - - - - - - - - -
        for (size_t i = 0; i < puntos.size(); ++i) {
            const auto& punto = puntos[i];
            // Calcular x.w + b
            float decision = 0.0;
            for (size_t j = 0; j < punto.caracteristicas.size(); ++j) {
                decision += punto.caracteristicas[j] * weights[j];
            }
            decision += bias;

            // Verificar si se viola la condición de margen
            if (punto.clase * decision <= 0) {
                // Actualizar pesos y bias
                for (size_t j = 0; j < weights.size(); ++j) {
                    weights[j] += learningRate * (punto.clase * punto.caracteristicas[j] - regularizationParam * weights[j]);
                }
                bias += learningRate * punto.clase;
            } else {
                correct++;
            }
        //- - - - - - - - - - - - - - - - - - - - - - - -

            if (mostrarProgreso) {
                // Simular espera para visualizar la barra de progreso
                std::this_thread::sleep_for(std::chrono::milliseconds(5));

                // Mostrar progreso
                int interval = puntos.size() / 10 == 0 ? 1 : puntos.size() / 10; // Evitar división por cero
                if (i % interval == 0 || i == puntos.size() - 1) { // Muestra progreso cada 10% y al final
                    int progress = static_cast<int>(std::round((i + 1) * 100.0 / puntos.size()));
                    int barWidth = 20;
                    int pos = barWidth * progress / 100;

                    // Obtener el uso de memoria
                    struct rusage usage;
                    getrusage(RUSAGE_SELF, &usage);
                    long memoryUsage = usage.ru_maxrss; // Uso máximo de memoria residente

                    std::cout << "\rEpoch " << epoch + 1 << "/" << epochs
                              << " [";
                    for (int k = 0; k < barWidth; ++k) {
                        if (k < pos) std::cout << "=";
                        else if (k == pos) std::cout << ">";
                        else std::cout << " ";
                    }
                    std::cout << "] " << std::setw(3) << progress << "% "
                              << "Correctos: " << correct << " de " << puntos.size()
                              << std::flush;
                }
            }
        }
        // Fin de la época
        if (mostrarProgreso) {
            std::cout << "\rEpoch " << epoch + 1 << "/" << epochs << " [====================] " << ANSI_COLOR_GREEN << "100% Correctos: " << correct << " de " << puntos.size() <<" (NO violan la condición de margen)" << ANSI_COLOR_RESET <<" { " << "Sesgo (Bias): " << bias <<" }" << std::endl;
        }
    }

    auto end = std::chrono::high_resolution_clock::now(); // Marca de tiempo de fin
    std::chrono::duration<double> duration = end - start; // Duración del entrenamiento

    // Obtener el uso de memoria final
    struct rusage usage;
    getrusage(RUSAGE_SELF, &usage);
    long memoryUsage = usage.ru_maxrss; // Uso máximo de memoria residente

    if (mostrarProgreso) {
        std::cout << "\nRESUMEN CONSUMO ENTRENAMIENTO\n";
        std::cout << " - Entrenamiento completo en " << duration.count() << " segundos." << std::endl;
        std::cout << " - Uso máximo de memoria: " << memoryUsage << " KB" << std::endl << std::endl;
    } else {
        std::cout << "Modelo entrenado " << ANSI_COLOR_GREEN <<"OK." << ANSI_COLOR_RESET << std::endl << std::endl;
    }
}

int SVM::clasificar(const Punto& puntoNuevo) const {
    float decision = 0.0;
    for (size_t i = 0; i < puntoNuevo.caracteristicas.size(); ++i) {
        decision += puntoNuevo.caracteristicas[i] * weights[i];
    }
    decision += bias;
    return (decision >= 0) ? 1 : -1;
}

void SVM::imprimirParametros(const std::vector<std::string>& etiquetas, bool mostarResumen) const {
    if (mostarResumen){
        std::cout << "\nRESUMEN DE LOS PARÁMETROS" << "\n----------------------------" << std::endl;
        for (size_t i = 0; i < weights.size(); ++i) {
            std::cout << " w" << i << " {" << etiquetas[i] << "}: " << weights[i] << std::endl;
        }
        std::cout << " B: " << bias << std::endl << std::endl;
    } else {
        std::cout << "Modelo entrenado " << ANSI_COLOR_GREEN <<"OK." << ANSI_COLOR_RESET << std::endl << std::endl;
    }
}


Writing /content/src/SVM.cpp


In [20]:
!g++ -c /content/src/SVM.cpp -o /content/src/SVM

***Ahora se construira el código principal que une todo: Punto + Datos + Modelo***

### Nombre del programa principal: main.cpp

In [21]:
%%writefile /content/src/main.cpp
#include <iostream>
#include <vector>
#include <string>
#include <limits>
#include "Datos.h"
#include "SVM.h"


// Colores para la salida
const std::string rojo = "\033[41;30m";
const std::string verde = "\033[42;30m";
const std::string reset = "\033[0m";

void ingresarYClasificar(SVM& svm, const std::vector<std::string>& etiquetas);
void parsear_argumentos(int argc, char* argv[], std::string& fuente_datos, float& tasa_aprendizaje, float& overfitting, int& epocas, bool& echo);


int main(int argc, char* argv[]) {

    // Variables para almacenar los valores de los argumentos y valores por defecto
    std::string fuente_datos;  //dataser con datos de conocimiento
    float tasa_aprendizaje = 0.001; //control de la velocidad de aprendizaje de modelo
    float overfitting = 0.001; //parámetro de regularización
    int epocas = 5;  //iteraciones del modelo
    bool echo = false;  //Bandera para ver o no el entrenamiento

    try {
        // Parsear argumentos de línea de comandos
        parsear_argumentos(argc, argv, fuente_datos, tasa_aprendizaje, overfitting, epocas, echo);

        // Imprimir información del programa
        std::cout << "Ejecución del programa...\n\n";
        std::cout << "DATOS DE LA INSTANCIA DEL PROGRAMA" << "\n----------------------------------\n";
        std::cout << " - Información del programa:" << std::endl;
        std::cout << " - Fuente de datos: " << fuente_datos << std::endl;
        std::cout << " - Tasa de aprendizaje: " << tasa_aprendizaje << std::endl;
        std::cout << " - Overfitting: " << overfitting << std::endl;
        std::cout << " - Epocas: " << epocas << std::endl;
        std::cout << " - Echo: " << (echo ? "true" : "false") << std::endl << std::endl;

        //Cargar el Dataset
        Datos lector;
        std::vector<Punto> puntos = lector.leerDatosCSV(fuente_datos);
        // Verificar que se leyeron datos correctamente
        if (puntos.empty() || lector.etiquetas.empty()) {
            std::cerr << rojo << "No se pudieron leer los datos del archivo CSV." << reset << std::endl;
            return 1;
        }

        // Crear el modelo SVM
        int numCaracteristicas = puntos[0].caracteristicas.size(); //cantidad de variables o atributos que describen la instancia en un conjunto de datos.
        SVM svm(tasa_aprendizaje, overfitting, numCaracteristicas);

        // Entrenar el modelo SVM
        svm.entrenar(puntos, epocas, echo);

        // Mostrar los pesos y el sesgo del hiperplano separador
        svm.imprimirParametros(lector.etiquetas, echo);

        // Ingresar y clasificar un nuevo punto de datos
        ingresarYClasificar(svm, lector.etiquetas);

    } catch (const std::runtime_error& e) {
        std::cerr << "Error: " << e.what() << std::endl;
        return 1;
    }

    return 0;
}

//=============================================================================
void ingresarYClasificar(SVM& svm, const std::vector<std::string>& etiquetas) {
    std::cout << "\nPRUEBAS\n" << "-------\n";

    bool otraClasificacion = true;
    while (otraClasificacion) {
        // Ingresar las características del nuevo punto
        std::cout << "Ingrese las características del nuevo punto:" << std::endl;
        std::vector<float> nuevasCaracteristicas(etiquetas.size());

        for (size_t i = 0; i < etiquetas.size(); ++i) {
            std::cout << "  " << etiquetas[i] << ": ";
            while (!(std::cin >> nuevasCaracteristicas[i])) {
                std::cin.clear();
                std::cin.ignore(std::numeric_limits<std::streamsize>::max(), '\n');
                std::cout << "         " << "\033[41;30m" << "Valor inválido. (Intente de nuevo!)"<< "\033[0m " << std::endl << "  " << etiquetas[i] << ": ";
            }
        }

        Punto puntoNuevo(nuevasCaracteristicas, 0); // La clase es irrelevante para la predicción
        int clasePredicha = svm.clasificar(puntoNuevo);

        if (clasePredicha == 1) {
            std::cout << "Clase predicha para el nuevo punto: " << rojo << "Maligno" << reset << std::endl;
        } else {
            std::cout << "Clase predicha para el nuevo punto: " << verde << "Benigno" << reset << std::endl;
        }
        std::cout << "\033[0m";

        // Preguntar si el usuario desea otra clasificación
        char otraClasificacionChar;
        std::cout << "\n¿Desea otra clasificación? ("<<verde<<" [si] s "<<reset<<" - "<<rojo<<" [no] n "<<reset<<"): ";
        std::cin >> otraClasificacionChar;

        // Convertir la respuesta del usuario a un valor booleano
        otraClasificacion = !(otraClasificacionChar == 'n' || otraClasificacionChar == 'N');
    }
}

//============================================================================================================
// Función para parsear argumentos de línea de comandos
void parsear_argumentos(int argc, char* argv[], std::string& fuente_datos, float& tasa_aprendizaje, float& overfitting, int& epocas, bool& echo) {
  // Variables para almacenar argumentos
  std::string argumento;

  // Procesar cada argumento
  for (int i = 1; i < argc; ++i) {
    argumento = argv[i];

    // Fuente de datos
    if (argumento == "--fuente_datos") {
      if (i + 1 < argc) {
        fuente_datos = argv[i + 1];
        ++i;
      } else {
        throw std::runtime_error("Falta el argumento para --fuente_datos");
      }
    }

    // Tasa de aprendizaje
    else if (argumento == "--tasa_aprendizaje") {
      if (i + 1 < argc) {
        try {
          tasa_aprendizaje = std::stof(argv[i + 1]);
        } catch (const std::exception& e) {
          throw std::runtime_error("Valor inválido para --tasa_aprendizaje: " + std::string(e.what()));
        }
        ++i;
      } else {
        throw std::runtime_error("Falta el argumento para --tasa_aprendizaje");
      }
    }

    // Overfitting
    else if (argumento == "--overfitting") {
      if (i + 1 < argc) {
        try {
          overfitting = std::stof(argv[i + 1]);
        } catch (const std::exception& e) {
          throw std::runtime_error("Valor inválido para --overfitting: " + std::string(e.what()));
        }
        ++i;
      } else {
        throw std::runtime_error("Falta el argumento para --overfitting");
      }
    }

    // Epocas
    else if (argumento == "--epocas") {
      if (i + 1 < argc) {
        try {
          epocas = std::stoi(argv[i + 1]);
        } catch (const std::exception& e) {
          throw std::runtime_error("Valor inválido para --epocas: " + std::string(e.what()));
        }
        ++i;
      } else {
        throw std::runtime_error("Falta el argumento para --epocas");
      }
    }

    // Echo
    else if (argumento == "--echo") {
      echo = true;
    }

    // Argumento no reconocido
    else {
      throw std::runtime_error("Argumento no reconocido: " + argumento);
    }
  }
}


Writing /content/src/main.cpp


In [22]:
!g++ /content/src/main.cpp /content/src/Punto /content/src/Datos /content/src/SVM -o /content/src/demo

In [23]:
!/content/src/demo --fuente_datos /content/datos_completos.csv --tasa_aprendizaje 0.0001 --overfitting 0.0001 --epocas 10 --echo

Ejecución del programa...

DATOS DE LA INSTANCIA DEL PROGRAMA
----------------------------------
 - Información del programa:
 - Fuente de datos: /content/datos_completos.csv
 - Tasa de aprendizaje: 0.0001
 - Overfitting: 0.0001
 - Epocas: 10
 - Echo: true

ENTRENANDO EL MODELO
--------------------
Epoch 1/10 [====================] 100% Correctos: 401 de 569 (NO violan la condición de margen) { Sesgo (Bias): -0.006 }
Epoch 2/10 [====================] 100% Correctos: 438 de 569 (NO violan la condición de margen) { Sesgo (Bias): -0.00969999 }
Epoch 3/10 [====================] 100% Correctos: 446 de 569 (NO violan la condición de margen) { Sesgo (Bias): -0.0128 }
Epoch 4/10 [====================] 100% Correctos: 450 de 569 (NO violan la condición de margen) { Sesgo (Bias): -0.0157 }
Epoch 5/10 [====================] 100% Correctos: 484 de 569 (NO violan la condición de margen) { Sesgo (Bias): -0.0178 }
Epoch 6/10 [====================] 100% Correctos: 480 de 569 (NO violan la condición de

## SVM en Paralelo (OpenMP)

### Archivo de cabecera: SVMomp.h

In [24]:
%%writefile /content/src/SVMomp.h
#ifndef SVMomp_H
#define SVMomp_H

#include <vector>
#include <string>


#include "Punto.h"

class SVMomp {
private:
    std::vector<float> weights;
    float bias;
    float learningRate;
    float regularizationParam;

public:
    SVMomp(float learningRate, float regularizationParam, int numCaracteristicas);

    void entrenar(const std::vector<Punto>& puntos, int epochs, bool mostrarProgreso = false);
    int clasificar(const Punto& puntoNuevo) const;
    void imprimirParametros(const std::vector<std::string>& etiquetas, bool mostarResumen = false) const;
};

#endif


Writing /content/src/SVMomp.h


### Archivo de cuerpo: SVMomp.cpp

In [25]:
%%writefile /content/src/SVMomp.cpp
#include "SVMomp.h"
#include <iostream>
#include <iomanip>
#include <chrono>
#include <stdexcept>
#include <cmath>
#include <ctime>  // Para std::chrono::milliseconds
#include <sys/resource.h> // Para obtener el uso de memoria
#include <omp.h>

// Secuencia de escape para colores ANSI
#define ANSI_COLOR_GREEN   "\x1b[32m"
#define ANSI_COLOR_RESET   "\x1b[0m"

SVMomp::SVMomp(float learningRate, float regularizationParam, int numCaracteristicas)
    : learningRate(learningRate), regularizationParam(regularizationParam), bias(0.0) {
    if (numCaracteristicas <= 0) {
        throw std::invalid_argument("El número de características debe ser mayor que 0.");
    }
    weights = std::vector<float>(numCaracteristicas, 0.0); // Inicializar pesos para las características
}

void SVMomp::entrenar(const std::vector<Punto>& puntos, int epochs, bool mostrarProgreso) {
    #pragma omp parallel default(none) shared(puntos, epochs, weights, bias, learningRate, regularizationParam)
        //Directiva #pragma omp parallel:
        //Esta directiva crea un equipo de subprocesos (threads) y define una región paralela.
        //Cada hilo en el equipo ejecutará el bloque de código dentro de esta región paralela.
        //
        //default(none) para forzar la especificación explícita de las variables compartidas y privadas,
        //shared para especificar las variables compartidas

    if (epochs <= 0) {
        throw std::invalid_argument("El número de épocas debe ser mayor que 0.");
    }
    if (puntos.empty()) {
        throw std::invalid_argument("El conjunto de puntos no debe estar vacío.");
    }

    auto start = std::chrono::high_resolution_clock::now(); // Marca de tiempo de inicio

    if (mostrarProgreso) {
        std::cout << "ENTRENANDO EL MODELO" << "\n--------------------" << std::endl;
    }

    //ajustar el modelo por iteraciones
    for (int epoch = 0; epoch < epochs; ++epoch) {
        int correct = 0; // Para contar el número de clasificaciones correctas

        //- - - - - [ OPERACIONES MATETICAS DEL MODELO ] - - - - - - - - - -
        #pragma omp for
                //Directiva #pragma omp for:
                //Esta directiva se utiliza para distribuir un bucle entre los hilos en la región paralela.
                //Cada hilo ejecutará una porción del bucle de forma concurrente.
        for (size_t i = 0; i < puntos.size(); ++i) {
            const auto& punto = puntos[i];
            // Calcular x.w + b
            float decision = 0.0;
            #pragma omp simd reduction(+:decision)
                    //Directiva #pragma omp simd reduction(+:variable):
                    //Esta directiva se utiliza para vectorizar el bucle y realizar la reducción de la variable especificada.
                    //En el código, se utiliza para calcular la suma de las decisiones en paralelo.
            for (size_t j = 0; j < punto.caracteristicas.size(); ++j) {
                decision += punto.caracteristicas[j] * weights[j];
            }
            decision += bias;

            // Verificar si se viola la condición de margen
            if (punto.clase * decision <= 0) {
                #pragma omp critical
                        //Directiva #pragma omp critical:
                        //Esta directiva se utiliza para proteger una sección crítica del código, asegurando que solo un hilo pueda ejecutarla a la vez.
                        //En el código proporcionado, se utiliza para garantizar que la impresión de la información de progreso del entrenamiento (cout) se realice de forma segura, evitando la escritura simultánea de múltiples hilos en la salida estándar.
                // Actualizar pesos y bias
                for (size_t j = 0; j < weights.size(); ++j) {
                    weights[j] += learningRate * (punto.clase * punto.caracteristicas[j] - regularizationParam * weights[j]);
                }
                bias += learningRate * punto.clase;
            } else {
                correct++;
            }
        //- - - - - - - - - - - - - - - - - - - - - - - -

            if (mostrarProgreso) {
                // Mostrar progreso
                int interval = puntos.size() / 10 == 0 ? 1 : puntos.size() / 10; // Evitar división por cero
                if (i % interval == 0 || i == puntos.size() - 1) { // Muestra progreso cada 10% y al final
                    int progress = static_cast<int>(std::round((i + 1) * 100.0 / puntos.size()));
                    int barWidth = 20;
                    int pos = barWidth * progress / 100;

                    // Obtener el uso de memoria
                    struct rusage usage;
                    getrusage(RUSAGE_SELF, &usage);
                    long memoryUsage = usage.ru_maxrss; // Uso máximo de memoria residente

                    std::cout << "\rEpoch " << epoch + 1 << "/" << epochs
                              << " [";
                    for (int k = 0; k < barWidth; ++k) {
                        if (k < pos) std::cout << "=";
                        else if (k == pos) std::cout << ">";
                        else std::cout << " ";
                    }
                    std::cout << "] " << std::setw(3) << progress << "% "
                              << "Correctos: " << correct << " de " << puntos.size()
                              << std::flush;
                }
            }
        }
        // Fin de la época
        if (mostrarProgreso) {
            std::cout << "\rEpoch " << epoch + 1 << "/" << epochs << " [====================] " << ANSI_COLOR_GREEN << "100% Correctos: " << correct << " de " << puntos.size() <<" (NO violan la condición de margen)" << ANSI_COLOR_RESET <<" { " << "Sesgo (Bias): " << bias <<" }" << std::endl;
        }
    }

    auto end = std::chrono::high_resolution_clock::now(); // Marca de tiempo de fin
    std::chrono::duration<double> duration = end - start; // Duración del entrenamiento

    // Obtener el uso de memoria final
    struct rusage usage;
    getrusage(RUSAGE_SELF, &usage);
    long memoryUsage = usage.ru_maxrss; // Uso máximo de memoria residente

    if (mostrarProgreso) {
        std::cout << "\nRESUMEN CONSUMO ENTRENAMIENTO\n";
        std::cout << " - Entrenamiento completo en " << duration.count() << " segundos." << std::endl;
        std::cout << " - Uso máximo de memoria: " << memoryUsage << " KB" << std::endl << std::endl;
    } else {
        std::cout << "Modelo entrenado " << ANSI_COLOR_GREEN <<"OK." << ANSI_COLOR_RESET << std::endl << std::endl;
    }
}

int SVMomp::clasificar(const Punto& puntoNuevo) const {
    float decision = 0.0;
    for (size_t i = 0; i < puntoNuevo.caracteristicas.size(); ++i) {
        decision += puntoNuevo.caracteristicas[i] * weights[i];
    }
    decision += bias;
    return (decision >= 0) ? 1 : -1;
}

void SVMomp::imprimirParametros(const std::vector<std::string>& etiquetas, bool mostarResumen) const {
    if (mostarResumen){
        std::cout << "\nRESUMEN DE LOS PARÁMETROS" << "\n----------------------------" << std::endl;
        for (size_t i = 0; i < weights.size(); ++i) {
            std::cout << " w" << i << " {" << etiquetas[i] << "}: " << weights[i] << std::endl;
        }
        std::cout << " B: " << bias << std::endl << std::endl;
    } else {
        std::cout << "Modelo entrenado " << ANSI_COLOR_GREEN <<"OK." << ANSI_COLOR_RESET << std::endl << std::endl;
    }
}


Writing /content/src/SVMomp.cpp


In [26]:
!g++ -c /content/src/SVMomp.cpp -o /content/src/SVMomp

### Archivo de cuerpo: main_omp.cpp

In [27]:
%%writefile /content/src/main_omp.cpp
#include <iostream>
#include <vector>
#include <string>
#include <limits>
#include "Datos.h"
#include "SVMomp.h"


// Colores para la salida
const std::string rojo = "\033[41;30m";
const std::string verde = "\033[42;30m";
const std::string reset = "\033[0m";

void ingresarYClasificar(SVMomp& svm, const std::vector<std::string>& etiquetas);
void parsear_argumentos(int argc, char* argv[], std::string& fuente_datos, float& tasa_aprendizaje, float& overfitting, int& epocas, bool& echo);

int main(int argc, char* argv[]) {
    // Variables para almacenar los valores de los argumentos y valores por defecto
    std::string fuente_datos;  //dataser con datos de conocimiento
    float tasa_aprendizaje = 0.001; //control de la velocidad de aprendizaje de modelo
    float overfitting = 0.001; //parámetro de regularización
    int epocas = 5;  //iteraciones del modelo
    bool echo = false;  //Bandera para ver o no el entrenamiento

    try {
        // Parsear argumentos de línea de comandos
        parsear_argumentos(argc, argv, fuente_datos, tasa_aprendizaje, overfitting, epocas, echo);

        // Imprimir información del programa
        std::cout << "Ejecución del programa...\n\n";
        std::cout << "DATOS DE LA INSTANCIA DEL PROGRAMA" << "\n----------------------------------\n";
        std::cout << " - Fuente de datos: " << fuente_datos << std::endl;
        std::cout << " - Tasa de aprendizaje: " << tasa_aprendizaje << std::endl;
        std::cout << " - Overfitting: " << overfitting << std::endl;
        std::cout << " - Epocas: " << epocas << std::endl;
        std::cout << " - Echo: " << (echo ? "true" : "false") << std::endl << std::endl;

        Datos lector;
        std::vector<Punto> puntos = lector.leerDatosCSV(fuente_datos);

        // Verificar que se leyeron datos correctamente
        if (puntos.empty() || lector.etiquetas.empty()) {
            std::cerr << rojo << "No se pudieron leer los datos del archivo CSV." << reset << std::endl;
            return 1;
        }

        // Crear el modelo SVM
        //float tasaAprendizaje = 0.001; //control de la velocidad de aprendizaje de modelo
        //float overfitting = 0.01; //parámetro de regularización
        int numCaracteristicas = puntos[0].caracteristicas.size(); //cantidad de variables o atributos que describen la instancia en un conjunto de datos.
        SVMomp svm(tasa_aprendizaje, overfitting, numCaracteristicas);

        // Entrenar el modelo SVM
        svm.entrenar(puntos, epocas, echo);

        // Mostrar los pesos y el sesgo del hiperplano separador
        svm.imprimirParametros(lector.etiquetas, echo);

        // Ingresar y clasificar un nuevo punto de datos
        ingresarYClasificar(svm, lector.etiquetas);

    } catch (const std::runtime_error& e) {
        std::cerr << "Error: " << e.what() << std::endl;
        return 1;
    }

    return 0;
}

//=============================================================================
void ingresarYClasificar(SVMomp& svm, const std::vector<std::string>& etiquetas) {
    std::cout << "\nPRUEBAS\n" << "-------\n";

    bool otraClasificacion = true;
    while (otraClasificacion) {
        // Ingresar las características del nuevo punto
        std::cout << "Ingrese las características del nuevo punto:" << std::endl;
        std::vector<float> nuevasCaracteristicas(etiquetas.size());

        for (size_t i = 0; i < etiquetas.size(); ++i) {
            std::cout << "  " << etiquetas[i] << ": ";
            while (!(std::cin >> nuevasCaracteristicas[i])) {
                std::cin.clear();
                std::cin.ignore(std::numeric_limits<std::streamsize>::max(), '\n');
                std::cout << "         " << "\033[41;30m" << "Valor inválido. (Intente de nuevo!)"<< "\033[0m " << std::endl << "  " << etiquetas[i] << ": ";
            }
        }

        Punto puntoNuevo(nuevasCaracteristicas, 0); // La clase es irrelevante para la predicción
        int clasePredicha = svm.clasificar(puntoNuevo);

        // Colores para la salida
        const std::string rojo = "\033[41;30m";
        const std::string verde = "\033[42;30m";
        const std::string reset = "\033[0m";

        if (clasePredicha == 1) {
            std::cout << "Clase predicha para el nuevo punto: " << rojo << "Maligno" << reset << std::endl;
        } else {
            std::cout << "Clase predicha para el nuevo punto: " << verde << "Benigno" << reset << std::endl;
        }
        std::cout << "\033[0m";

        // Preguntar si el usuario desea otra clasificación
        char otraClasificacionChar;
        std::cout << "\n¿Desea otra clasificación? ("<<verde<<" [si] s "<<reset<<" - "<<rojo<<" [no] n "<<reset<<"): ";
        std::cin >> otraClasificacionChar;

        // Convertir la respuesta del usuario a un valor booleano
        otraClasificacion = !(otraClasificacionChar == 'n' || otraClasificacionChar == 'N');
    }
}

//============================================================================================================
// Función para parsear argumentos de línea de comandos
void parsear_argumentos(int argc, char* argv[], std::string& fuente_datos, float& tasa_aprendizaje, float& overfitting, int& epocas, bool& echo) {
  // Variables para almacenar argumentos
  std::string argumento;

  // Procesar cada argumento
  for (int i = 1; i < argc; ++i) {
    argumento = argv[i];

    // Fuente de datos
    if (argumento == "--fuente_datos") {
      if (i + 1 < argc) {
        fuente_datos = argv[i + 1];
        ++i;
      } else {
        throw std::runtime_error("Falta el argumento para --fuente_datos");
      }
    }

    // Tasa de aprendizaje
    else if (argumento == "--tasa_aprendizaje") {
      if (i + 1 < argc) {
        try {
          tasa_aprendizaje = std::stof(argv[i + 1]);
        } catch (const std::exception& e) {
          throw std::runtime_error("Valor inválido para --tasa_aprendizaje: " + std::string(e.what()));
        }
        ++i;
      } else {
        throw std::runtime_error("Falta el argumento para --tasa_aprendizaje");
      }
    }

    // Overfitting
    else if (argumento == "--overfitting") {
      if (i + 1 < argc) {
        try {
          overfitting = std::stof(argv[i + 1]);
        } catch (const std::exception& e) {
          throw std::runtime_error("Valor inválido para --overfitting: " + std::string(e.what()));
        }
        ++i;
      } else {
        throw std::runtime_error("Falta el argumento para --overfitting");
      }
    }

    // Epocas
    else if (argumento == "--epocas") {
      if (i + 1 < argc) {
        try {
          epocas = std::stoi(argv[i + 1]);
        } catch (const std::exception& e) {
          throw std::runtime_error("Valor inválido para --epocas: " + std::string(e.what()));
        }
        ++i;
      } else {
        throw std::runtime_error("Falta el argumento para --epocas");
      }
    }

    // Echo
    else if (argumento == "--echo") {
      echo = true;
    }

    // Argumento no reconocido
    else {
      throw std::runtime_error("Argumento no reconocido: " + argumento);
    }
  }
}

Writing /content/src/main_omp.cpp


In [28]:
!mkdir -p /content/out_omp

In [29]:
!g++ -c /content/src/Punto.cpp -o /content/out_omp/Punto -fopenmp
!g++ -c /content/src/Datos.cpp -o /content/out_omp/Datos -fopenmp
!g++ -c /content/src/SVMomp.cpp -o /content/out_omp/SVMomp -fopenmp

In [30]:
!g++ /content/src/main_omp.cpp /content/out_omp/Punto /content/out_omp/Datos /content/out_omp/SVMomp -o /content/out_omp/demo -fopenmp

In [31]:
!/content/out_omp/demo --fuente_datos /content/datos_completos.csv --tasa_aprendizaje 0.0001 --overfitting 0.0001 --epocas 10 --echo

Ejecución del programa...

DATOS DE LA INSTANCIA DEL PROGRAMA
----------------------------------
 - Fuente de datos: /content/datos_completos.csv
 - Tasa de aprendizaje: 0.0001
 - Overfitting: 0.0001
 - Epocas: 10
 - Echo: true

ENTRENANDO EL MODELO
--------------------
Epoch 1/10 [====================] 100% Correctos: 401 de 569 (NO violan la condición de margen) { Sesgo (Bias): -0.006 }
Epoch 2/10 [====================] 100% Correctos: 438 de 569 (NO violan la condición de margen) { Sesgo (Bias): -0.00969999 }
Epoch 3/10 [====================] 100% Correctos: 446 de 569 (NO violan la condición de margen) { Sesgo (Bias): -0.0128 }
Epoch 4/10 [====================] 100% Correctos: 450 de 569 (NO violan la condición de margen) { Sesgo (Bias): -0.0157 }
Epoch 5/10 [====================] 100% Correctos: 484 de 569 (NO violan la condición de margen) { Sesgo (Bias): -0.0178 }
Epoch 6/10 [====================] 100% Correctos: 480 de 569 (NO violan la condición de margen) { Sesgo (Bias): -0.0